# 05 — Indexation de Caltech-101

Les chemins sont relatifs à data/101_ObjectCategories. Le dossier BACKGROUND_Google est exclu pour cette expérience.

**Dépendances de la construction de l’index**

- Pillow ouvre les JPEG et produit les représentations HSV et grise. NumPy forme et sauvegarde les matrices. scikit-image calcule les motifs LBP.
- La cellule installe ces paquets dans le noyau choisi. L’indexeur affiche son avancement et ses résultats sous forme de texte et écrit une archive NumPy dans le dossier des données.

In [ ]:
%pip install pillow numpy scikit-image

**Calculer deux descripteurs compatibles avec la recherche**

- `signatures(chemin)` reçoit un chemin et retourne un couple de tableaux. Le premier contient **512 proportions de couleur** issues d’un histogramme HSV à 8 × 8 × 8 cases. Le second contient **10 proportions de texture** issues d’un LBP uniforme à huit voisins et rayon 1.
- Le traitement de chaque image commence par la même transformation en RGB carré de 256 × 256. La couleur est calculée sur le HSV et la texture sur les niveaux de gris.
- La normalisation séparée donne une somme de 1 à chaque descripteur. Le notebook de recherche doit reprendre la même taille et les mêmes paramètres. Si tu modifies les descripteurs, régénère l’index avant d’utiliser les nouvelles signatures de requête.

In [ ]:
from pathlib import Path
from time import perf_counter
from uuid import uuid4
import numpy as np
from PIL import Image
from skimage.feature import local_binary_pattern

def signatures(chemin):
    with Image.open(chemin) as image:
        rgb = image.convert("RGB").resize((256, 256), Image.Resampling.LANCZOS)
    hsv = np.asarray(rgb.convert("HSV"))
    gris = np.asarray(rgb.convert("L"))
    couleur, _ = np.histogramdd(hsv.reshape(-1, 3), bins=(8, 8, 8),
                               range=((0, 256), (0, 256), (0, 256)))
    lbp = local_binary_pattern(gris, 8, 1, method="uniform")
    texture, _ = np.histogram(lbp, bins=np.arange(11))
    return (couleur / couleur.sum()).ravel(), texture / texture.sum()

**Construire, contrôler puis enregistrer l’index**

- `glob("*/*.jpg")` parcourt les catégories. Le filtre exclut `BACKGROUND_Google`. Une erreur de lecture `OSError` est imprimée et la boucle continue. Seules les images traitées avec succès alimentent les listes. Une ligne représente la même image dans les quatre tableaux : chemin relatif à la base, catégorie, signature HSV et signature LBP.
- On transforme les descripteurs en matrices `float32` de formes **(N, 512)** et **(N, 10)**. Les assertions vérifient les formes et la somme de chaque ligne. Les chemins relatifs permettent de déplacer le dossier du projet tout en conservant les mêmes chemins internes.
- `np.savez_compressed` écrit les tableaux sous quatre noms dans un fichier temporaire. On relit ce fichier avec `allow_pickle=False` et on vérifie les tableaux avant que `replace` mette en place l’index définitif. Le bloc `finally` nettoie le fichier temporaire même si une opération échoue. Le chronomètre inclut l’extraction, la sauvegarde et cette vérification.

In [ ]:
debut = perf_counter()
base = Path("data/101_ObjectCategories")
images = sorted(p for p in base.glob("*/*.jpg") if p.parent.name != "BACKGROUND_Google")
chemins, categories, couleurs, textures = [], [], [], []
for i, chemin in enumerate(images, 1):
    try:
        couleur, texture = signatures(chemin)
    except OSError as erreur:
        print(f"Image ignoree : {chemin}: {erreur}")
    else:
        chemins.append(chemin.relative_to(base).as_posix())
        categories.append(chemin.parent.name)
        couleurs.append(couleur)
        textures.append(texture)
    if i % 1000 == 0:
        print(f"{i}/{len(images)} images examinees")

if not chemins:
    raise ValueError(f"Aucune image indexable dans {base.resolve()}")
couleurs = np.asarray(couleurs, dtype=np.float32)
textures = np.asarray(textures, dtype=np.float32)
assert couleurs.shape == (len(chemins), 512) and textures.shape == (len(chemins), 10)
assert np.allclose(couleurs.sum(axis=1), 1) and np.allclose(textures.sum(axis=1), 1)

sortie = Path("data/index_caltech101.npz")
temporaire = sortie.with_suffix(f".{uuid4().hex}.npz")
donnees = dict(chemins=chemins, categories=categories, couleurs=couleurs, textures=textures)
try:
    np.savez_compressed(temporaire, **donnees)
    with np.load(temporaire, allow_pickle=False) as index:
        assert all(np.array_equal(index[cle], valeur) for cle, valeur in donnees.items())
    temporaire.replace(sortie)
finally:
    temporaire.unlink(missing_ok=True)
print(f"{len(chemins)} images indexees, {len(images)-len(chemins)} ignorees, {perf_counter()-debut:.1f} s sur CPU.")
print(sortie.resolve())